RAM check
!free -h
Disk check
!df -h
GPU check (agar enable kiya ho to)
!nvidia-smi

## Phase 2 → Lesson 6: Data Preprocessing Pipeline
### 📘 Short Theory — Why This Matters?
prepare it. This step alone determines whether your model is good or garbage.



Garbage in = Garbage out. No matter how powerful your model is — bad data kills it.

Data preprocessing includes:

* Handling missing values

   * nan par mathematics apply nhi krskty ,
  
   * Numeric data (Age/Salary) ke liye Mean/Median best hota hai taake overall distribution kharab na ho.
   
   * Categorical data (City/Dept) ke liye Most Frequent (Mode) use hota hai.
* Encoding categorical data

  * Computers sirf numbers samajhte hain.texxt nhi

  * Label Encoding: Jaham order matter karta hai (e.g., Junior=0, Mid=1, Senior=2).

  * One-Hot Encoding: Jahan koi rank nahi hoti (e.g., Karachi, Lahore, Islamabad). Isme har city ka naya column (0/1) banta hai taake model kisi city ko doosri se "greater" na samjhe.

* Feature scaling

  * Theory: Salary (50,000) Age (25) se bohot badi value hai. Agar scale na karein, toh model samjhega Salary 2000x zyada important hai.

  * StandardScaler: Mean ko 0 aur Standard Deviation ko 1 kar deta hai (Best for general ML).

  * MinMaxScaler: Sub values ko 0 se 1 ke beech fit kar deta hai.
* Splitting data properly
4. Sklearn Pipeline & Data Leakage Prevention:

Theory: Practice Exercise 7 ka answer yeh hai ke agar hum pure dataset par scaler ya imputer fit karenge, toh test data ki information train data me "leak" ho jayegi (Data Leakage).

Logic: Rule humesha yeh hai ke Fit sirf training data par hoga, aur Transform test data par.


All in one clean pipeline. 🧠

### concep 1 - The Problem with Raw data

In [ ]:
import numpy as np
import pandas as pd

# Typical messy real world dataset
data = {
    'Age':        [25, None, 35, 45, None, 28, 52, 38],
    'Salary':     [30000, 45000, None, 60000, 52000, None, 80000, 42000],
    'City':       ['Karachi', 'Lahore', 'Karachi', None, 'Islamabad', 'Lahore', 'Karachi', 'Islamabad'],
    'Experience': ['Junior', 'Senior', 'Junior', 'Senior', 'Mid', 'Mid', 'Senior', 'Junior'],
    'Hired':      [0, 1, 0, 1, 1, 0, 1, 0]}

df= pd.DataFrame(data)
print(df,"\n")
print(df.describe())
print(f'\n Missing values : \n{df.isnull().sum()}\n')


    Age   Salary       City Experience  Hired
0  25.0  30000.0    Karachi     Junior      0
1   NaN  45000.0     Lahore     Senior      1
2  35.0      NaN    Karachi     Junior      0
3  45.0  60000.0       None     Senior      1
4   NaN  52000.0  Islamabad        Mid      1
5  28.0      NaN     Lahore        Mid      0
6  52.0  80000.0    Karachi     Senior      1
7  38.0  42000.0  Islamabad     Junior      0 

             Age        Salary     Hired
count   6.000000      6.000000  8.000000
mean   37.166667  51500.000000  0.500000
std    10.186592  17201.744098  0.534522
min    25.000000  30000.000000  0.000000
25%    29.750000  42750.000000  0.000000
50%    36.500000  48500.000000  0.500000
75%    43.250000  58000.000000  1.000000
max    52.000000  80000.000000  1.000000

 Missing values : 
Age           2
Salary        2
City          1
Experience    0
Hired         0
dtype: int64



## 🧠 Concept 2 — Handling Missing Values (Imputation)

* SimpleImputer(strategy='mean') → replaces NaN with column mean

* strategy='most_frequent' → replaces NaN with most common value

* fit_transform() → learns the fill value AND applies it, sb sy fit value ko calculate krta ha , learn krta or transform replace krta ,data pr apply krta hha

* ravel() → flattens 2D output to 1D for categorical
because skleanr accept 2d and dataframe in panda series so accept one col as sereis

In [ ]:
from sklearn.impute import SimpleImputer

# For numerical columns --fill with mean
num_imputer = SimpleImputer(strategy='mean')
df['Age']= num_imputer.fit_transform(df[['Age']])
df['Salary']= num_imputer.fit_transform(df[['Salary']])

# For categorical columns -- fill with most frequent
cat_imputer = SimpleImputer(strategy='most_frequent')
df['City']= cat_imputer.fit_transform(df[['City']]).ravel()

print(df.isnull().sum()) # should all be 0 now
print(df)

Age           0
Salary        0
City          1
Experience    0
Hired         0
dtype: int64
         Age   Salary       City Experience  Hired
0  25.000000  30000.0    Karachi     Junior      0
1  37.166667  45000.0     Lahore     Senior      1
2  35.000000  51500.0    Karachi     Junior      0
3  45.000000  60000.0       None     Senior      1
4  37.166667  52000.0  Islamabad        Mid      1
5  28.000000  51500.0     Lahore        Mid      0
6  52.000000  80000.0    Karachi     Senior      1
7  38.000000  42000.0  Islamabad     Junior      0


## CONCEPT 3 - Encoding Categorical Data
ML models only understand numbers — not text. So we convert categories to numbers.

Label Encoding — for ordered categories (Junior < Mid < Senior):

* LabelEncoder() → converts text to numbers 0,1,2...
get_dummies() → creates separate binary column for each category for ordinal 0,1,2

* One Hot is better for cities because Karachi(2) > Lahore(1) would imply ranking which
doesn't exist,it is like for nominal , which can not be calculated and it will make col then flag it
```
pd.get_dummies())

Yahan kisi city ko number dene ke bajaye har city ka alag column (0 ya 1) bana diya jata hai:
```
1. prefix='City' Kya Karta Hai?

* Matlab: Naye banne wale columns ke naam ke shuru mein yeh prefix (tag) laga deta hai.

 Kyun zaroori hai?

Agar hum prefix='City' lagayein, toh naye columns ke naam banenge:

City_Islamabad

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df['Experience_encoded']= le.fit_transform(df['Experience'])
print(df[['Experience','Experience_encoded']])
#junior == 0,mid == 1 , senior == 2
# One Hot Encoding — for unordered categories (cities have no order):
df=pd.get_dummies(df,columns=['City'],prefix='City')
print(df.head()) # to check first 5



  Experience  Experience_encoded
0     Junior                   0
1     Senior                   2
2     Junior                   0
3     Senior                   2
4        Mid                   1
5        Mid                   1
6     Senior                   2
7     Junior                   0
         Age   Salary Experience  Hired  Experience_encoded  City_Islamabad  \
0  25.000000  30000.0     Junior      0                   0           False   
1  37.166667  45000.0     Senior      1                   2           False   
2  35.000000  51500.0     Junior      0                   0           False   
3  45.000000  60000.0     Senior      1                   2           False   
4  37.166667  52000.0        Mid      1                   1            True   

   City_Karachi  City_Lahore  
0          True        False  
1         False         True  
2          True        False  
3         False        False  
4         False        False  


## 🧠 Concept 4 — Feature Scaling

Age ranges 25-52. Salary ranges 30000-80000. The model thinks Salary is 1000x more important just because the numbers are bigger. Scaling fixes this.


* StandardScaler() → subtracts mean, divides by std → centers around 0
* MinMaxScaler() → scales to 0-1 range
StandardScaler used when data has outliers
MinMaxScaler used when you need values between 0 and 1

1. StandardScaler (Standardization)Yeh data ko is tarah adjust karta hai ke uska Mean ($\mu$) = 0 aur Standard Deviation ($\sigma$) = 1 ho jaye.isko z-score normalization bolty ha

2. MinMaxScaler (Normalization)
Yeh dataset ki saari values ko 0 se 1 ke beech (ya -1 se 1) squeeze/fit kar deta hai.

more details notes me page code FeaSc00


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# StandardScaler → mean=0, std=1 (most common)
scaler = StandardScaler()
df[['Age', 'Salary']] = scaler.fit_transform(df[['Age', 'Salary']])
print(df[['Age', 'Salary']].head())

# MinMaxScaler → scales to range 0-1
mm_scaler = MinMaxScaler()
# mm_scaler.fit_transform(df[['Age', 'Salary']])

        Age    Salary
0 -1.510785 -1.580979
1  0.000000 -0.477970
2 -0.269044  0.000000
3  0.972697  0.625038
4  0.000000  0.036767


## Concept 5 --Full Pipeline with sklearn

Instead of doing all steps manually — sklearn Pipeline chains everything together:

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler # Added StandardScaler import
from sklearn.metrics import accuracy_score # Added accuracy_score import

# Prepare clean data
X = df.drop(['Hired', 'Experience'], axis=1) # Drop 'Experience' column as it's not numerical
y = df['Hired']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Build pipeline
pipeline = Pipeline([
    ('scaler',  StandardScaler()),
    ('model',   RandomForestClassifier(n_estimators=100, random_state=42))
])
v# Fit Command: Step 1, 2, aur 3 ek saath execute ho jayenge!z
# Train entire pipeline at once
pipeline.fit(X_train, y_train)

# Predict
y_pred = pipeline.predict(X_test)
print(f'Pipeline Accuracy: {accuracy_score(y_test, y_pred):.2f}')

Pipeline Accuracy: 1.00


auto doing by make pipelines from sklearn instead of doing manually

Step-by-Step Code Structure
Aap apne saare manual steps ko is tarah ek preprocessor mein fit kar sakte hain:

```
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

# 1. Columns Define Karein
num_cols = ['Age', 'Salary']
nominal_cols = ['City']
ordinal_cols = ['Experience']  # Junior < Mid < Senior

# 2. Numerical Pipeline (Fill NaN + Scaling)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

# 3. Nominal Pipeline (Fill NaN + One-Hot Encoding)
nominal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# 4. Ordinal Pipeline (Fill NaN + Rank Encoding)
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Junior', 'Mid', 'Senior']]))
])

# 5. ColumnTransformer (Preprocessor Engine)
preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('nom', nominal_pipeline, nominal_cols),
    ('ord', ordinal_pipeline, ordinal_cols)
])
```
Analogy: Yeh bilkul aisa hai jaise aap ek Machine ka Blueprint tayyar kar rahe ho:
* Pehla pipe (num_pipeline) $\rightarrow$ Sirf Age aur Salary par chale.
* Dusra pipe (nominal_pipeline) $\rightarrow$ Sirf City par chale.
* Teesra pipe (ordinal_pipeline) $\rightarrow$ Sirf Experience par chale.

x and y already uper splt dropes test and train ho chuky ha isly wo steps yha nhi ha

***Tarika 1: Master Pipeline***

 (Sab se Best aur Professional Tarika)Aap preprocessor ke sath end mein Model jodh kar ek Master Pipeline bana dete hain. Tab model target ($y$) se train hota hai:


 ```
 # Master Pipeline: Preprocessor Engine + Model
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),              # 1. Clean & Encode (X ke liye)
    ('model', RandomForestClassifier())          # 2. ML Model
])

# FIT Command: Yeh X_train ko clean karega aur y_train se model train kar dega!
full_pipeline.fit(X_train, y_train)

# Predict Command: X_test ko auto-clean kar ke predictions nikal dega
y_pred = full_pipeline.predict(X_test)
 ```
***Tarika 2: Manually Direct Preprocessor Use Karna***
Agar aap alag se steps dekhna chahte hain:
```
# Pehle sirf X_train ko clean/transform kiya
X_train_clean = preprocessor.fit_transform(X_train)

<!-- preprocessor ne pehle X_train ke columns ka Mean, Mode, Std Dev
 wagerah seekha (fit), aur phir usi ke mutabiq X_train ki missing values
bhari, scaling ki, aur encoding karke naya X_train_clean bana diya. -->

X_test_clean = preprocessor.transform(X_test)

<!-- Kya hua: Notice karein yahan fit nahi lagaya, sirf transform lagaya!

Kyun? Kyunki test data se humein kuch seekhna nahi hai (Data Leakage se
bachne ke liye). Hum ne test data par bilkul wahi rules apply kiye jo X_train se seekhe the. -->

# Phir trained model mein y_train diya
model = RandomForestClassifier()
model.fit(X_train_clean, y_train)  # Yahan y_train diya train karne ke liye!
```



## Practise Set
Exercise 1: Print missing value count per column.

Exercise 2: Fill missing Age and Salary with mean. Fill missing Department with most frequent.

Exercise 3: Label encode the Level column.

Exercise 4: One hot encode the Department column.

Exercise 5: Scale Age and Salary using StandardScaler.

Exercise 6: Print the final clean DataFrame — how many columns does it have now?

Exercise 7 (thinking): Why do we fit the scaler on training data only and not the full dataset?

In [11]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.preprocessing import StandardScaler

data = {
    'Age':        [23, None, 31, 45, 28, None, 35, 42, 29, 38],
    'Salary':     [25000, 40000, None, 70000, 35000, 55000, None, 65000, 30000, 48000],
    'Department': ['IT', 'HR', 'IT', 'Finance', None, 'HR', 'Finance', 'IT', 'HR', 'Finance'],
    'Level':      ['Junior', 'Senior', 'Mid', 'Senior', 'Junior', 'Mid', 'Senior', 'Mid', 'Junior', 'Senior'],
    'Promoted':   [0, 1, 0, 1, 0, 1, 1, 1, 0, 1]}


df=pd.DataFrame(data)
print(df,'\n')
#Task 1
print(f'missing values count per col\n')
print(df.isnull().sum(),'\n')
Age_num=SimpleImputer(strategy='mean')
df['Age']=Age_num.fit_transform(df[['Age']])
df['Salary']=Age_num.fit_transform(df[['Salary']])
#second task
# Convert None to np.nan in 'Department' column to ensure correct imputation
df['Department'] = df['Department'].replace({None: np.nan})
cat_imputer = SimpleImputer(strategy='most_frequent')
df['Department']=cat_imputer.fit_transform(df[['Department']]).ravel()
#thrd task
le=LabelEncoder()
df['Level_encoded']=le.fit_transform(df['Level'])
print(df[['Level','Level_encoded']])
#4th task
df=pd.get_dummies(df,columns=['Department'],prefix='Department')

scaler=StandardScaler()
df[['Age','Salary']]=scaler.fit_transform(df[['Age','Salary']])
df=df.drop('Level',axis=1)
arr=df.shape[1]
print(f'the no of cols after cleaning data is : ',arr)
print(df.head())
print(df.isnull().sum(),'\n')


    Age   Salary Department   Level  Promoted
0  23.0  25000.0         IT  Junior         0
1   NaN  40000.0         HR  Senior         1
2  31.0      NaN         IT     Mid         0
3  45.0  70000.0    Finance  Senior         1
4  28.0  35000.0       None  Junior         0
5   NaN  55000.0         HR     Mid         1
6  35.0      NaN    Finance  Senior         1
7  42.0  65000.0         IT     Mid         1
8  29.0  30000.0         HR  Junior         0
9  38.0  48000.0    Finance  Senior         1 

missing values count per col

Age           2
Salary        2
Department    1
Level         0
Promoted      0
dtype: int64 

    Level  Level_encoded
0  Junior              0
1  Senior              2
2     Mid              1
3  Senior              2
4  Junior              0
5     Mid              1
6  Senior              2
7     Mid              1
8  Junior              0
9  Senior              2
the no of cols after cleaning data is :  7
        Age    Salary  Promoted  Level_encoded  D